> 📘 **FES** · Fonaments d'Enginyeria de Software amb Python

# 15 · Herència i polimorfisme

**Ampliació opcional.** Una hora de teoria i una hora de pràctica addicionals al calendari obligatori. Requereix classes, composició i excepcions.

## Objectius d'aprenentatge

- Especialitzar sensors amb herència i `super()`.
- Reutilitzar una interfície comuna amb polimorfisme.
- Aplicar la convenció d'ús intern `_lectures`.
- Retornar còpies i conservar l'estat quan es rebutja una lectura.

Tots els sensors són simulats. Executa les cel·les en ordre des d'un kernel net.

## Classe base i interfície comuna

`Sensor` conserva la identificació i l'històric. Les operacions públiques són `registrar`, `ultima_lectura`, `mitjana` i `obtenir_lectures`. La classe manté la responsabilitat de validar les mesures abans de desar-les.

El prefix `_` indica ús intern per convenció. Python permet accedir a `_lectures`, però el codi client utilitza els mètodes públics. `_validar` agrupa les comprovacions comunes de tipus numèric i finitud. `registrar` crida aquest mètode abans d'`append`.

`obtenir_lectures` retorna una còpia de la llista. Les mesures són nombres immutables, de manera que aquesta còpia permet consultar i modificar el resultat sense canviar l'històric intern.

In [ ]:
from math import isfinite


class Sensor:
    """Model general d'un sensor amb un històric de nombres finits."""

    def __init__(self, nom, unitat):
        """Crea un sensor amb identificació i estat propi."""
        self.nom = nom
        self.unitat = unitat
        # El prefix _ indica ús intern per convenció. Python permet
        # accedir-hi, però el codi client utilitza els mètodes públics.
        self._lectures = []

    def _validar(self, valor):
        """Comprova les condicions comunes abans de canviar l'estat."""
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            raise ValueError("La lectura ha de ser un nombre.")
        if isinstance(valor, float) and not isfinite(valor):
            raise ValueError("La lectura ha de ser finita.")

    def registrar(self, valor):
        """Valida una lectura i després l'afegeix a l'històric."""
        # self._validar selecciona el mètode de la instància concreta.
        self._validar(valor)
        self._lectures.append(valor)

    def ultima_lectura(self):
        """Retorna l'última lectura o None si l'històric és buit."""
        if not self._lectures:
            return None
        return self._lectures[-1]

    def mitjana(self):
        """Retorna la mitjana aritmètica o None si no hi ha lectures."""
        if not self._lectures:
            return None
        return sum(self._lectures) / len(self._lectures)

    def obtenir_lectures(self):
        """Retorna una còpia de la llista per consultar l'històric."""
        # Les mesures són nombres immutables. Una còpia de la llista
        # permet modificar la llista retornada sense alterar l'original.
        return self._lectures.copy()

## Herència i especialització

La sintaxi `class SensorTemperatura(Sensor)` declara una subclasse. Un sensor de temperatura és un tipus de sensor i hereta els mètodes de la classe base. El constructor especialitzat fixa la unitat amb `super().__init__`.

Cada subclasse redefineix `_validar`. Primer reutilitza la validació comuna amb `super()._validar` i després comprova el seu rang. En aquest model simulat, temperatura admet de −40 a 85 °C i humitat de 0 a 100 %, amb els extrems inclosos. Aquests rangs formen part de l'especificació de l'exercici.

In [ ]:
class SensorTemperatura(Sensor):
    """Sensor simulat de temperatura entre -40 i 85 °C, extrems inclosos."""

    def __init__(self, nom):
        """Reutilitza la inicialització i fixa la unitat de temperatura."""
        super().__init__(nom, "°C")

    def _validar(self, valor):
        """Combina la validació numèrica i el rang de temperatura."""
        super()._validar(valor)
        if not -40 <= valor <= 85:
            raise ValueError("La temperatura ha d'estar entre -40 i 85 °C.")


class SensorHumitat(Sensor):
    """Sensor simulat d'humitat relativa entre 0 i 100 %, inclosos."""

    def __init__(self, nom):
        """Reutilitza la inicialització i fixa la unitat d'humitat."""
        super().__init__(nom, "%")

    def _validar(self, valor):
        """Combina la validació numèrica i el rang d'humitat."""
        super()._validar(valor)
        if not 0 <= valor <= 100:
            raise ValueError("La humitat ha d'estar entre 0 i 100 %.")

## Una mateixa crida per a tipus diferents

El polimorfisme permet utilitzar una mateixa operació amb objectes de tipus diferents. El bucle següent consulta els dos sensors mitjançant els mateixos mètodes.

Quan executem `temperatura.registrar(20.0)`, Python utilitza el `registrar` heretat. Dins d'aquest mètode, `self._validar` selecciona la versió de `SensorTemperatura`. Aquesta versió crida la validació comuna i comprova el rang. La lectura s'afegeix un cop s'han superat totes les comprovacions.

In [ ]:
temperatura = SensorTemperatura("T1")
humitat = SensorHumitat("H1")
for valor in [20.0, 24.0]:
    temperatura.registrar(valor)
for valor in [40.0, 60.0]:
    humitat.registrar(valor)

sensors = [temperatura, humitat]
for sensor in sensors:
    print(sensor.nom, sensor.ultima_lectura(), sensor.mitjana(),
          sensor.unitat)
# T1 24.0 22.0 °C
# H1 60.0 50.0 %

## Errors i extrems del rang

La validació precedeix qualsevol canvi de l'històric. Rebutjar una lectura conserva l'últim valor vàlid. `bool`, text, `nan` i infinits es rebutgen amb `ValueError` abans de comparar rangs.

Prediu quins intents s'acceptaran. Després identifica si cada rebuig s'origina a la classe base o a la subclasse.

In [ ]:
limits = SensorTemperatura("T2")
for valor in [-40, 85, -41, 86, True, "25", float("nan"),
              float("inf")]:
    try:
        limits.registrar(valor)
    except ValueError as error:
        print(error)
print(limits.obtenir_lectures())  # [-40, 85]

## Encapsulació bàsica i còpia

L'encapsulació agrupa estat i operacions i defineix com s'utilitza l'objecte. La convenció `_lectures` comunica que el registre es gestiona amb els mètodes. `obtenir_lectures()` retorna una llista nova amb les mateixes mesures.

Prediu els dos resultats. Explica per què afegir 999.0 a la llista retornada conserva l'última lectura de T1.

In [ ]:
copia = temperatura.obtenir_lectures()
copia.append(999.0)
print(copia)  # [20.0, 24.0, 999.0]
print(temperatura.obtenir_lectures())  # [20.0, 24.0]
print(temperatura.ultima_lectura())  # 24.0

## El comportament comú també inclou casos buits

Les subclasses hereten les consultes buides de `Sensor`. L'última lectura i la mitjana retornen `None` mentre no s'hagi registrat cap valor. L'històric consultat és una llista buida. Aquest comportament coherent facilita reutilitzar el mateix codi client i les mateixes proves amb els diferents tipus.

In [ ]:
buit = SensorHumitat("H0")
print(buit.ultima_lectura())  # None
print(buit.mitjana())  # None
print(buit.obtenir_lectures())  # []

## Relació amb el projecte i pràctica

Al bloc 12, `SistemaMonitoratge` conté sensors mitjançant composició d'objectes. En aquest bloc, `SensorTemperatura` i `SensorHumitat` especialitzen `Sensor` mitjançant herència. Podem combinar les dues relacions. El sistema coordina les instàncies i cada tipus determina les validacions del seu rang.

Conservar una interfície comuna permet afegir tipus sense reescriure els bucles que consulten els sensors. Els exercicis 1–3 treballen construcció, validació i encapsulació. El repte opcional 4 afegeix un sensor de tensió.

[Enunciats](01_herencia_exercicis.py) · [Solucions](01_herencia_solucions.py)